In [2]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav.columns


Index(['amfi_code', 'date', 'nav'], dtype='str')

In [7]:
nav.dtypes

amfi_code      int64
date             str
nav          float64
dtype: object

In [8]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [9]:
nav.duplicated().sum()

np.int64(0)

In [11]:
nav["date"] = pd.to_datetime(nav["date"])

In [12]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [13]:
nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
       .ffill()
)


In [14]:
nav = nav.drop_duplicates()

In [15]:
nav.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

In [19]:
nav.columns

Index(['amfi_code', 'date', 'nav'], dtype='str')

In [20]:
nav.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [22]:
txn = pd.read_csv("../data/raw/08_investor_transactions.csv")

txn.columns

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='str')

In [23]:
txn["transaction_type"] = (
    txn["transaction_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

txn["transaction_type"] = txn["transaction_type"].replace({
    "SIP PAYMENT":"SIP",
    "SIP":"SIP",
    "LUMPSUM":"LUMPSUM",
    "REDEEM":"REDEMPTION",
    "REDEMPTION":"REDEMPTION"
})

In [26]:
invalid_amount = txn[txn["amount_inr"] <= 0]

print("Invalid Amount Records:", len(invalid_amount))

Invalid Amount Records: 0


In [27]:
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"],
    errors="coerce"
)

In [28]:
txn["kyc_status"] = txn["kyc_status"].str.upper()

valid_kyc = ["YES","NO","PENDING"]

invalid_kyc = txn[
    ~txn["kyc_status"].isin(valid_kyc)
]

print("Invalid KYC:", len(invalid_kyc))

Invalid KYC: 30146


In [29]:
txn.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

In [31]:
perf = pd.read_csv(
    "../data/raw/07_scheme_performance.csv"
)

perf.columns

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='str')

In [35]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

In [36]:
for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

In [37]:
for col in return_cols:
    anomalies = perf[
        perf[col].abs() > 100
    ]

    print(col, len(anomalies))

return_1yr_pct 0
return_3yr_pct 0
return_5yr_pct 0


In [39]:
bad_expense = perf[
    (perf["expense_ratio_pct"] < 0.1)
    |
    (perf["expense_ratio_pct"] > 2.5)
]

print("Bad Expense Ratio:", len(bad_expense))

Bad Expense Ratio: 0


In [40]:
perf.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

In [41]:
from sqlalchemy import create_engine
import pandas as pd

# Create SQLite database
engine = create_engine(
    "sqlite:///mutual_fund_analytics.db"
)

In [42]:
nav = pd.read_csv(
    "../data/processed/nav_history_clean.csv"
)

nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [43]:
nav.to_sql(
    "nav_history",
    engine,
    if_exists="replace",
    index=False
)

print("NAV table loaded successfully")

NAV table loaded successfully


In [44]:
import sqlite3

conn = sqlite3.connect(
    "mutual_fund_analytics.db"
)

query = """
SELECT COUNT(*)
FROM nav_history
"""

print(
    pd.read_sql(query, conn)
)

   COUNT(*)
0     46000


In [45]:
txn = pd.read_csv(
    "../data/processed/investor_transactions_clean.csv"
)

txn.to_sql(
    "investor_transactions",
    engine,
    if_exists="replace",
    index=False
)

32778

In [46]:
perf = pd.read_csv(
    "../data/processed/scheme_performance_clean.csv"
)

perf.to_sql(
    "scheme_performance",
    engine,
    if_exists="replace",
    index=False
)

40

In [47]:
txn = pd.read_csv("../data/processed/investor_transactions_clean.csv")

txn.to_sql(
    "investor_transactions",
    engine,
    if_exists="replace",
    index=False
)

print("investor_transactions loaded")

investor_transactions loaded


In [48]:
perf = pd.read_csv("../data/processed/scheme_performance_clean.csv")

perf.to_sql(
    "scheme_performance",
    engine,
    if_exists="replace",
    index=False
)

print("scheme_performance loaded")

scheme_performance loaded


In [49]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mutual_fund_analytics.db")

tables = [
    "nav_history",
    "investor_transactions",
    "scheme_performance"
]

for table in tables:
    query = f"SELECT COUNT(*) as rows FROM {table}"
    print(table)
    print(pd.read_sql(query, conn))
    print("-" * 30)

nav_history
    rows
0  46000
------------------------------
investor_transactions
    rows
0  32778
------------------------------
scheme_performance
   rows
0    40
------------------------------


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mutual_fund_analytics.db")

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("mutual_fund_analytics.db")

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,nav_history
1,investor_transactions
2,scheme_performance


In [11]:
query = """
SELECT *
FROM nav_history
LIMIT 10;
"""

pd.read_sql(query, conn)

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-10,510.7136
6,100016,2022-01-11,513.5542
7,100016,2022-01-12,512.3195
8,100016,2022-01-13,510.2445
9,100016,2022-01-14,514.3636


In [3]:
import pandas as pd
fund = pd.read_csv("../data/raw/01_fund_master.csv")

print(fund.shape)
print(fund.isnull().sum())
print("Duplicates:", fund.duplicated().sum())

(40, 15)
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64
Duplicates: 0


In [4]:
fund.columns = fund.columns.str.lower().str.strip()

fund = fund.drop_duplicates()

for col in fund.select_dtypes(include="object").columns:
    fund[col] = fund[col].astype(str).str.strip()

fund.to_csv(
    "../data/processed/fund_master_clean.csv",
    index=False
)

/var/folders/dg/f4y65v4d7wn1j55pg__yvvkm0000gn/T/ipykernel_64011/105922854.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in fund.select_dtypes(include="object").columns:


In [5]:
import pandas as pd
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")

print(aum.shape)
print(aum.isnull().sum())
print("Duplicates:", aum.duplicated().sum())

(90, 5)
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64
Duplicates: 0


In [6]:
aum = aum.drop_duplicates()

numeric_cols = aum.select_dtypes(include=["int64","float64"]).columns

for col in numeric_cols:
    aum[col] = pd.to_numeric(aum[col], errors="coerce")

aum.to_csv(
    "../data/processed/aum_by_fund_house_clean.csv",
    index=False
)

In [7]:
import pandas as pd
sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")

print(sip.shape)
print(sip.isnull().sum())

(48, 6)
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64


In [8]:
sip = sip.drop_duplicates()

for col in sip.columns:
    if "date" in col.lower() or "month" in col.lower():
        sip[col] = pd.to_datetime(sip[col], errors="coerce")

sip.to_csv(
    "../data/processed/monthly_sip_inflows_clean.csv",
    index=False
)

In [9]:
import pandas as pd
cat = pd.read_csv("../data/raw/05_category_inflows.csv")

cat = cat.drop_duplicates()

for col in cat.select_dtypes(include="object").columns:
    cat[col] = cat[col].astype(str).str.strip()

cat.to_csv(
    "../data/processed/category_inflows_clean.csv",
    index=False
)

/var/folders/dg/f4y65v4d7wn1j55pg__yvvkm0000gn/T/ipykernel_64011/4238269879.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in cat.select_dtypes(include="object").columns:


In [10]:
folio = pd.read_csv("../data/raw/06_industry_folio_count.csv")

folio = folio.drop_duplicates()

folio.to_csv(
    "../data/processed/industry_folio_count_clean.csv",
    index=False
)

In [14]:
import pandas as pd
holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")

holdings = holdings.drop_duplicates()

for col in holdings.select_dtypes(include="object").columns:
    holdings[col] = holdings[col].astype(str).str.strip()

holdings.to_csv(
    "../data/processed/portfolio_holdings_clean.csv",
    index=False
)

/var/folders/dg/f4y65v4d7wn1j55pg__yvvkm0000gn/T/ipykernel_64011/1826155887.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in holdings.select_dtypes(include="object").columns:


In [15]:
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

benchmark = benchmark.drop_duplicates()

for col in benchmark.columns:
    if "date" in col.lower():
        benchmark[col] = pd.to_datetime(
            benchmark[col],
            errors="coerce"
        )

benchmark.to_csv(
    "../data/processed/benchmark_indices_clean.csv",
    index=False
)

In [20]:
from sqlalchemy import create_engine
import os

# Create database
engine = create_engine("sqlite:///bluestock_mf.db")

# Force creation by opening a connection
with engine.connect():
    pass

print("Database created!")
print("Current folder:", os.getcwd())
print("Database exists:", os.path.exists("bluestock_mf.db"))

Database created!
Current folder: /Users/syedshaju/Desktop/Mutual_Fund_Analytics/notebooks
Database exists: True


In [21]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///bluestock_mf.db")

files = {
    "fund_master": "../data/processed/fund_master_clean.csv",
    "nav_history": "../data/processed/nav_history_clean.csv",
    "aum_by_fund_house": "../data/processed/aum_by_fund_house_clean.csv",
    "monthly_sip_inflows": "../data/processed/monthly_sip_inflows_clean.csv",
    "category_inflows": "../data/processed/category_inflows_clean.csv",
    "industry_folio_count": "../data/processed/industry_folio_count_clean.csv",
    "scheme_performance": "../data/processed/scheme_performance_clean.csv",
    "investor_transactions": "../data/processed/investor_transactions_clean.csv",
    "portfolio_holdings": "../data/processed/portfolio_holdings_clean.csv",
    "benchmark_indices": "../data/processed/benchmark_indices_clean.csv"
}

for table_name, file_path in files.items():
    df = pd.read_csv(file_path)
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )
    print(f"Loaded: {table_name} ({len(df)} rows)")

print("All tables loaded successfully!")

Loaded: fund_master (40 rows)
Loaded: nav_history (46000 rows)
Loaded: aum_by_fund_house (90 rows)
Loaded: monthly_sip_inflows (48 rows)
Loaded: category_inflows (144 rows)
Loaded: industry_folio_count (21 rows)
Loaded: scheme_performance (40 rows)
Loaded: investor_transactions (32778 rows)
Loaded: portfolio_holdings (322 rows)
Loaded: benchmark_indices (8050 rows)
All tables loaded successfully!


In [22]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn)

,name
0,aum_by_fund_house
1,benchmark_indices
2,category_inflows
3,fund_master
4,industry_folio_count
5,investor_transactions
6,monthly_sip_inflows
7,nav_history
8,portfolio_holdings
9,scheme_performance


In [23]:
tables = [
    "fund_master",
    "nav_history",
    "aum_by_fund_house",
    "monthly_sip_inflows",
    "category_inflows",
    "industry_folio_count",
    "scheme_performance",
    "investor_transactions",
    "portfolio_holdings",
    "benchmark_indices"
]

for table in tables:
    query = f"SELECT COUNT(*) AS rows FROM {table}"
    count = pd.read_sql(query, conn)
    print(f"\n{table}")
    print(count)


fund_master
   rows
0    40

nav_history
    rows
0  46000

aum_by_fund_house
   rows
0    90

monthly_sip_inflows
   rows
0    48

category_inflows
   rows
0   144

industry_folio_count
   rows
0    21

scheme_performance
   rows
0    40

investor_transactions
    rows
0  32778

portfolio_holdings
   rows
0   322

benchmark_indices
   rows
0  8050


In [24]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,fund_master
1,nav_history
2,aum_by_fund_house
3,monthly_sip_inflows
4,category_inflows
5,industry_folio_count
6,scheme_performance
7,investor_transactions
8,portfolio_holdings
9,benchmark_indices


In [ ]:
print(fund_master.columns)
print(nav_history.columns)
print(aum_by_fund_house.columns)
print(monthly_sip_inflows.columns)
print(category_inflows.columns)
print(scheme_performance.columns)
print(fund_master.columns)
print(nav_history.columns)
print(aum_by_fund_house.columns)
print(monthly_sip_inflows.columns)
print(category_inflows.columns)
print(scheme_performance.columns)
print(investor_transactions.columns)